# Initial Analysis and Cleaning

Load raw sales data, inspect schema and quality, clean transactions, and save `merged_transactions.csv`.

This notebook was extracted from `Combined.ipynb` and organized as a standalone stage.

## Notebook Guide
This notebook loads the source retail datasets, checks schema quality, and produces a cleaned transaction table used by the downstream analysis.


In [1]:
from pathlib import Path

PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / 'dataset'
GENERATED_DIR = PROJECT_ROOT / 'generated'
FIGURES_DIR = PROJECT_ROOT / 'figures'

GENERATED_DIR.mkdir(exist_ok=True)
FIGURES_DIR.mkdir(exist_ok=True)

print(f'Project root: {PROJECT_ROOT}')
print(f'Dataset dir: {DATA_DIR}')
print(f'Generated dir: {GENERATED_DIR}')
print(f'Figures dir: {FIGURES_DIR}')


Project root: /Users/ponnimuthukumarasamy/Downloads/Prosol Dataset Complete/Sales-Data-Analysis
Dataset dir: /Users/ponnimuthukumarasamy/Downloads/Prosol Dataset Complete/Sales-Data-Analysis/dataset
Generated dir: /Users/ponnimuthukumarasamy/Downloads/Prosol Dataset Complete/Sales-Data-Analysis/generated
Figures dir: /Users/ponnimuthukumarasamy/Downloads/Prosol Dataset Complete/Sales-Data-Analysis/figures


In [2]:
"""
Customer Segmentation Analysis - Retail Transaction Data
=========================================================

This notebook provides a complete workflow for customer segmentation
based on transactional data from TOKEN, VENTE, ARTICLE, and MAGASIN tables.

Project Goals:
1. Identify distinct customer segments based on purchasing behavior
2. Characterize each segment's habits and preferences
3. Discover sales opportunities (upselling, cross-selling, optimization)
4. Provide actionable business insights
"""

"\nCustomer Segmentation Analysis - Retail Transaction Data\n=========================================================\n\nThis notebook provides a complete workflow for customer segmentation\nbased on transactional data from TOKEN, VENTE, ARTICLE, and MAGASIN tables.\n\nProject Goals:\n1. Identify distinct customer segments based on purchasing behavior\n2. Characterize each segment's habits and preferences\n3. Discover sales opportunities (upselling, cross-selling, optimization)\n4. Provide actionable business insights\n"

In [3]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

In [4]:
# For clustering
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN
from sklearn.decomposition import PCA
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from sklearn.metrics import silhouette_score, davies_bouldin_score

# For association rules
from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.preprocessing import TransactionEncoder

In [5]:
# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', lambda x: '%.3f' % x)

# Visualization settings
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

In [6]:


# Create session
print("\n[STEP 1.1] Loading CSV files...")

try:
    df_token = pd.read_csv(DATA_DIR / 'TOKEN.csv')
    df_vente = pd.read_csv(DATA_DIR / 'VENTE.csv')
    df_article = pd.read_csv(DATA_DIR / 'ARTICLE.csv')
    df_magasin = pd.read_csv(DATA_DIR / 'MAGASIN.csv')

    
    for df in [df_token, df_vente, df_article, df_magasin]:
        df.columns = [c.strip() for c in df.columns]
    
    print(f"✓ TOKEN loaded: {df_token.shape}")
    print(f"✓ VENTE loaded: {df_vente.shape}")
    print(f"✓ ARTICLE loaded: {df_article.shape}")
    print(f"✓ MAGASIN loaded: {df_magasin.shape}")
    
except FileNotFoundError as e:
    print(f"ERROR: {e}")
    print("Please update the file paths in the code to match your CSV locations.")

# Load tables
# df_vente = session.table("POC_ECOLE.SALES.VENTE").to_pandas()
# df_token = session.table("POC_ECOLE.SALES.TOKEN").to_pandas()
# df_article = session.table("POC_ECOLE.SALES.ARTICLE").to_pandas()
# df_magasin = session.table("POC_ECOLE.SALES.MAGASIN").to_pandas()

# Clean column names
for df in [df_token, df_vente, df_article, df_magasin]:
    df.columns = [c.strip() for c in df.columns]

print(f"✓ TOKEN loaded: {df_token.shape}")
print(f"✓ VENTE loaded: {df_vente.shape}")
print(f"✓ ARTICLE loaded: {df_article.shape}")
print(f"✓ MAGASIN loaded: {df_magasin.shape}")


[STEP 1.1] Loading CSV files...
✓ TOKEN loaded: (174007, 2)
✓ VENTE loaded: (32879372, 11)
✓ ARTICLE loaded: (2312, 7)
✓ MAGASIN loaded: (66, 9)
✓ TOKEN loaded: (174007, 2)
✓ VENTE loaded: (32879372, 11)
✓ ARTICLE loaded: (2312, 7)
✓ MAGASIN loaded: (66, 9)


In [7]:
df_magasin.info()

<class 'pandas.DataFrame'>
RangeIndex: 66 entries, 0 to 65
Data columns (total 9 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   ID_MAG_TIERS                  66 non-null     int64  
 1   ID_LIEU_DE_VENTE              66 non-null     int64  
 2   LB_DESIGNATION_LIEU_DE_VENTE  66 non-null     str    
 3   LB_ADRESSE                    66 non-null     str    
 4   CD_POSTAL                     66 non-null     int64  
 5   LB_VILLE                      66 non-null     str    
 6   LATITUDE                      66 non-null     float64
 7   LONGITUDE                     66 non-null     float64
 8   DT_OUVERTURE                  66 non-null     str    
dtypes: float64(2), int64(3), str(4)
memory usage: 4.8 KB


In [8]:
print("\n[STEP 1.2] Initial Data Exploration")

def explore_dataframe(df, name):
    print(f"\n{name} Table:")
    print(f"  Shape: {df.shape}")
    print(f"  Columns: {list(df.columns)}")
    print(f"  Memory usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
    print(f"  Missing values:\n{df.isnull().sum()[df.isnull().sum() > 0]}")
    print(f"\n  First few rows:")
    print(df.head(3))
    return df.info()

# Explore each table
for df, name in [(df_token, 'TOKEN'), (df_vente, 'VENTE'),
                  (df_article, 'ARTICLE'), (df_magasin, 'MAGASIN')]:
    explore_dataframe(df, name)


[STEP 1.2] Initial Data Exploration

TOKEN Table:
  Shape: (174007, 2)
  Columns: ['NO_TOKEN_CB', 'CD_TICKET_UNIQUE']
  Memory usage: 20.08 MB
  Missing values:
Series([], dtype: int64)

  First few rows:
                                         NO_TOKEN_CB     CD_TICKET_UNIQUE
0  A33FB7919EEAB875A8AF4493286211B29F49B0746939EF...  4949207601061614291
1  2755B58F8A00BA3FA60E9E69A1951194F048FDF5FB0C2C...  4945949002201926787
2  7B6704A84131A41FF8F2CB121FD2517FE65F52B714B4C4...  4964269402201934474
<class 'pandas.DataFrame'>
RangeIndex: 174007 entries, 0 to 174006
Data columns (total 2 columns):
 #   Column            Non-Null Count   Dtype
---  ------            --------------   -----
 0   NO_TOKEN_CB       174007 non-null  str  
 1   CD_TICKET_UNIQUE  174007 non-null  int64
dtypes: int64(1), str(1)
memory usage: 2.7 MB

VENTE Table:
  Shape: (32879372, 11)
  Columns: ['CD_TICKET_UNIQUE', 'DT_VENTE', 'TS_VENTE', 'ID_MAG_TIERS', 'ID_ARTICLE', 'MT_TTC_NET', 'TX_TVA', 'QT_UVC', 'QT_POIDS',

In [9]:
df_vente.info()

<class 'pandas.DataFrame'>
RangeIndex: 32879372 entries, 0 to 32879371
Data columns (total 11 columns):
 #   Column              Dtype  
---  ------              -----  
 0   CD_TICKET_UNIQUE    int64  
 1   DT_VENTE            str    
 2   TS_VENTE            str    
 3   ID_MAG_TIERS        int64  
 4   ID_ARTICLE          str    
 5   MT_TTC_NET          float64
 6   TX_TVA              float64
 7   QT_UVC              float64
 8   QT_POIDS            float64
 9   IS_PROMO_NATIONALE  int64  
 10  IS_PROMO_MAGASIN    int64  
dtypes: float64(4), int64(4), str(3)
memory usage: 2.7 GB


## STEP 1.3: Data Type Conversion and Preparation
Convert key columns to analysis-ready types before applying any business rules.


In [10]:
# ============================================================================
# STEP 1.3: Data Type Conversion and Preparation
# ============================================================================

print("\n[STEP 1.3] Converting data types...")

# Convert date and timestamp fields
df_vente['DT_VENTE'] = pd.to_datetime(df_vente['DT_VENTE'], errors='coerce')
df_vente['TS_VENTE'] = pd.to_datetime(df_vente['TS_VENTE'], format='%Y-%m-%d-%H.%M.%S.%f', errors='coerce')

# Extract time-based features
df_vente['HOUR'] = df_vente['TS_VENTE'].dt.hour
df_vente['DAY_OF_WEEK'] = df_vente['TS_VENTE'].dt.dayofweek  # 0=Monday, 6=Sunday
df_vente['DAY_NAME'] = df_vente['TS_VENTE'].dt.day_name()
df_vente['MONTH'] = df_vente['DT_VENTE'].dt.month
df_vente['YEAR'] = df_vente['DT_VENTE'].dt.year
df_vente['IS_WEEKEND'] = df_vente['DAY_OF_WEEK'].isin([5, 6]).astype(int)

print("✓ Date/time fields converted and features extracted")


[STEP 1.3] Converting data types...
✓ Date/time fields converted and features extracted


In [11]:
df_vente['TS_VENTE'].isnull().sum()

np.int64(0)

## STEP 1.4: Data Quality Assessment
Profile missing values, invalid values, and suspicious records that need to be handled.


In [12]:
# ============================================================================
# STEP 1.4: Data Quality Assessment
# ============================================================================

print("\n[STEP 1.4] Data Quality Assessment")
print("-" * 80)

# Check for duplicates
print(f"Duplicate tickets in VENTE: {df_vente['CD_TICKET_UNIQUE'].duplicated().sum()}")
print(f"Duplicate tokens in TOKEN: {df_token['NO_TOKEN_CB'].duplicated().sum()}")

# Check key relationships
print(f"\nUnique customers (tokens): {df_token['NO_TOKEN_CB'].nunique()}")
print(f"Unique tickets: {df_vente['CD_TICKET_UNIQUE'].nunique()}")
print(f"Unique articles: {df_vente['ID_ARTICLE'].nunique()}")

# Check Store IDs (Handling ID_LIEU_DE_VENTE vs ID_MAGASIN mismatch possibility)
if 'ID_LIEU_DE_VENTE' in df_magasin.columns:
    print(f"Unique stores (Magasin Table): {df_magasin['ID_LIEU_DE_VENTE'].nunique()}")

# Date range
print(f"\nDate range: {df_vente['DT_VENTE'].min()} to {df_vente['DT_VENTE'].max()}")  # It said in the email that 'the sales of the full year 2025'

# Statistics
print(f"\nTransaction amount statistics (Raw):")
print(df_vente['MT_TTC_NET'].describe())

print(f"\nQuantity statistics (Raw):")
print(df_vente[['QT_UVC', 'QT_POIDS']].describe())

# Check for negative values (returns)
print(f"\nNegative amounts (returns): {(df_vente['MT_TTC_NET'] < 0).sum()}")
print(f"Zero amounts: {(df_vente['MT_TTC_NET'] == 0).sum()}")


[STEP 1.4] Data Quality Assessment
--------------------------------------------------------------------------------
Duplicate tickets in VENTE: 28597998
Duplicate tokens in TOKEN: 52859

Unique customers (tokens): 121148
Unique tickets: 4281374
Unique articles: 4415
Unique stores (Magasin Table): 11

Date range: 2025-01-02 00:00:00 to 2025-12-31 00:00:00

Transaction amount statistics (Raw):
count   32879372.000
mean           3.335
std            2.967
min         -285.360
25%            1.670
50%            2.650
75%            3.990
max          725.200
Name: MT_TTC_NET, dtype: float64

Quantity statistics (Raw):
            QT_UVC     QT_POIDS
count 32879372.000 32879372.000
mean         1.118        0.345
std          0.944        0.706
min        -93.120      -93.127
25%          0.940        0.000
50%          1.000        0.000
75%          1.000        0.499
max        222.350      222.350

Negative amounts (returns): 8632
Zero amounts: 95543


## PART 2: Data Cleaning and PreProcessing
Apply the cleaning rules that remove unreliable transactions and standardize the working dataset.


In [13]:
# ============================================================================
# PART 2: Data Cleaning and PreProcessing
# ============================================================================

print("\n" + "=" * 80)
print("PART 2: Data Cleaning and PreProcessing")
print("=" * 80)


PART 2: Data Cleaning and PreProcessing


In [14]:
print('Data Issues')
'''
1. Prices less than 0 or 0
2. Quantities less than 0
'''

Data Issues


'\n1. Prices less than 0 or 0\n2. Quantities less than 0\n'

In [15]:
print("\nMerging tables...")

# 1. Merge TOKEN + VENTE
if 'CD_TICKET UNIQUE' in df_vente.columns:
    df_vente.rename(columns={'CD_TICKET UNIQUE': 'CD_TICKET_UNIQUE'}, inplace=True)

df_transactions = df_token.merge(
    df_vente,
    on='CD_TICKET_UNIQUE',
    how='inner'
)

# 2. Merge with ARTICLE
df_transactions = df_transactions.merge(
    df_article,
    on='ID_ARTICLE',
    how='left'
)

# 3. Merge + MAGASIN
print("Merging Store data (ID + Location info)...")


store_cols_to_merge = [
    'ID_MAG_TIERS',
    'ID_LIEU_DE_VENTE',
    'LB_DESIGNATION_LIEU_DE_VENTE',
    'LB_ADRESSE',
    'CD_POSTAL',
    'LB_VILLE',
    'LATITUDE',
    'LONGITUDE',
    'DT_OUVERTURE'
]

df_transactions = df_transactions.merge(
    df_magasin[store_cols_to_merge],
    on='ID_MAG_TIERS',
    how='left',
    indicator=True
)

print(f"✓ Merged shape: {df_transactions.shape}")
print("✓ Store info mapped: Location & Coordinates added.")

# Check merge quality BEFORE dropping _merge column
print('\nCheck merge quality:')
print(df_transactions['_merge'].value_counts())

missing_store_count = df_transactions[df_transactions['_merge'] == 'left_only'].shape[0]
print(f"Missing store info count: {missing_store_count}")

# Check other missing info
print(f"Missing bill info: {df_transactions['DT_VENTE'].isnull().sum()}")
if 'LB_METIER' in df_transactions.columns:
    print(f"Missing article info: {df_transactions['LB_METIER'].isnull().sum()}")

# Drop merge indicator
df_transactions.drop(columns=['_merge'], inplace=True)


Merging tables...
Merging Store data (ID + Location info)...
✓ Merged shape: (1305191, 33)
✓ Store info mapped: Location & Coordinates added.

Check merge quality:
_merge
both          1305191
left_only           0
right_only          0
Name: count, dtype: int64
Missing store info count: 0
Missing bill info: 0
Missing article info: 0


In [16]:
print("\n[STEP 2.1] Data Cleaning...")

# 1. Remove Refunds (Negative Price or Quantity)
# Note: We do NOT filter for QT_POIDS > 0 separately, as that would delete non-weighted items.
initial_rows = len(df_transactions)
df_transactions = df_transactions[
    (df_transactions['MT_TTC_NET'] >= 0) &
    (df_transactions['QT_UVC'] >= 0)
].copy()

rows_removed = initial_rows - len(df_transactions)
print(f"✓ Removed {rows_removed} refund/negative rows.")

# 2. Handle Zero Prices
zero_prices = (df_transactions['MT_TTC_NET'] == 0).sum()
print(f"  Note: There are {zero_prices} transactions with 0 price (100% discount/error).")


[STEP 2.1] Data Cleaning...
✓ Removed 845 refund/negative rows.
  Note: There are 2828 transactions with 0 price (100% discount/error).


In [17]:
print("\n[STEP 2.2] Creating Helper Columns...")

# 1. Unified Quantity Metric (Unit + Weight)
df_transactions['TOTAL_QTY'] = df_transactions['QT_UVC'].fillna(0) + df_transactions['QT_POIDS'].fillna(0)

# 2. Quantity on Promotion (for accurate promo %)
df_transactions['QTY_PROMO_NAT'] = df_transactions['TOTAL_QTY'] * df_transactions['IS_PROMO_NATIONALE']
df_transactions['QTY_PROMO_STORE'] = df_transactions['TOTAL_QTY'] * df_transactions['IS_PROMO_MAGASIN']


[STEP 2.2] Creating Helper Columns...


In [18]:
df_transactions['DT_VENTE'].max()

Timestamp('2025-12-23 00:00:00')

In [19]:
print("\n" + "=" * 80)
print("Data Cleaned and PreProcessed")
print("=" * 80)


Data Cleaned and PreProcessed


In [20]:
df_transactions.head()

,NO_TOKEN_CB,CD_TICKET_UNIQUE,DT_VENTE,TS_VENTE,ID_MAG_TIERS,ID_ARTICLE,MT_TTC_NET,TX_TVA,QT_UVC,QT_POIDS,IS_PROMO_NATIONALE,IS_PROMO_MAGASIN,HOUR,DAY_OF_WEEK,DAY_NAME,MONTH,YEAR,IS_WEEKEND,LB_METIER,LB_SOUS_RAYON,LB_FAMILLE,LB_SOUS_FAMILLE,CD_ARTICLE,LB_ARTICLE,ID_LIEU_DE_VENTE,LB_DESIGNATION_LIEU_DE_VENTE,LB_ADRESSE,CD_POSTAL,LB_VILLE,LATITUDE,LONGITUDE,DT_OUVERTURE,TOTAL_QTY,QTY_PROMO_NAT,QTY_PROMO_STORE
0,A33FB7919EEAB875A8AF4493286211B29F49B0746939EF...,4949207601061614291,2025-12-05,2025-12-05 08:46:00,79,100000000008209,1.090,0.055,0.550,0.550,0,0,8,4,Friday,12,2025,0,Fruits et légumes,Fruits et légumes,Carotte,Carotte,8209,Carotte Extra Vrac,106,Vaulx en Velin,2 RUE JEAN ET JOSEPHINE PEYRI,69120,VAULX EN VELIN,45.786,4.927,2007-05-16,1.100,0.000,0.000
1,A33FB7919EEAB875A8AF4493286211B29F49B0746939EF...,4949207601061614291,2025-12-05,2025-12-05 08:46:00,79,100000000009130,0.500,0.055,0.200,0.200,0,0,8,4,Friday,12,2025,0,Fruits et légumes,Fruits et légumes,Navet et Rave,Rave,9130,Rave,106,Vaulx en Velin,2 RUE JEAN ET JOSEPHINE PEYRI,69120,VAULX EN VELIN,45.786,4.927,2007-05-16,0.400,0.000,0.000
2,2755B58F8A00BA3FA60E9E69A1951194F048FDF5FB0C2C...,4945949002201926787,2025-12-01,2025-12-01 14:15:00,1296,100000000009179,0.990,0.055,1.000,0.000,0,0,14,0,Monday,12,2025,0,Fruits et légumes,Fruits et légumes,Herbes Aromatiques,Coriandre,9179,Coriandre Bouquet Colis,220,Wasquehal,10 rue Jean Jaures,59290,Wasquehal,50.680,3.138,2018-02-14,1.000,0.000,0.000
3,2755B58F8A00BA3FA60E9E69A1951194F048FDF5FB0C2C...,4945949002201926787,2025-12-01,2025-12-01 14:15:00,1296,100000000005706,1.560,0.055,0.190,0.195,0,0,14,0,Monday,12,2025,0,Fruits et légumes,Fruits et légumes,Agrumes,Citron,5706,Lime Non Traite,220,Wasquehal,10 rue Jean Jaures,59290,Wasquehal,50.680,3.138,2018-02-14,0.385,0.000,0.000
4,7B6704A84131A41FF8F2CB121FD2517FE65F52B714B4C4...,4964269402201934474,2025-12-22,2025-12-22 19:09:00,1297,80000000000C1HB,5.390,0.055,1.000,0.110,0,0,19,0,Monday,12,2025,0,Crémerie,Coupe / Corner,PPC,Tête de moine,C1HB,Fleurs de Tête de Moine AOP Signature Spielhofer,220,Wasquehal,10 rue Jean Jaures,59290,Wasquehal,50.680,3.138,2018-02-14,1.110,0.000,0.000


In [21]:
output_path = GENERATED_DIR / 'merged_transactions.csv'
df_transactions.to_csv(output_path, index=False)
print(f"✓ Saved merged transactions to '{output_path}'")


✓ Saved merged transactions to '/Users/ponnimuthukumarasamy/Downloads/Prosol Dataset Complete/Sales-Data-Analysis/generated/merged_transactions.csv'


## Results Summary

- Loaded and inspected the raw retail source tables before merging them into one transaction-level dataset.
- Removed refund or negative-value rows and highlighted the presence of zero-price transactions that may reflect discounts or data-entry issues.
- Added helper columns needed for downstream aggregation and confirmed the cleaned transaction timeline extends to late December 2025.
- Main output: `generated/merged_transactions.csv`.
